# TikTok Creator Intelligence

This notebook has two sections:

**Section A** — Install packages and launch the Streamlit web app 
**Section B** — Run the TikTok data scraper (separate, run after you have your msToken)

---

## Section A — Streamlit Web App

### Step 1 — Install all packages

This installs everything the app needs:
- `streamlit` — the web framework that runs your app in a browser
- `pandas` — for reading and processing the CSV data
- `plotly` — for the charts and graphs
- `nltk` — Natural Language Toolkit, provides the VADER sentiment analyser
- `scikit-learn` — provides TF-IDF for keyword extraction
- `openpyxl` — lets the app read Excel files
- `vaderSentiment` — the sentiment scoring library

Run this cell once. The `!` means 'run in terminal, not Python'.

In [ ]:
!python -m pip install streamlit pandas plotly nltk scikit-learn openpyxl vaderSentiment

### Step 2 — Download the VADER language data

VADER (the sentiment analyser) needs to download its word list once before it can work.
This cell does that download. Run it once.

In [ ]:
import nltk
nltk.download('vader_lexicon')
print('VADER lexicon downloaded successfully.')

### Step 3 — Launch the web app

This starts the Streamlit server and automatically opens the app in your browser.

**What to expect:**
- A browser tab opens at `http://localhost:8501`
- The app starts on the Login page
- Register a new account, then log in
- The Upload Data page loads with demo data already analysed
- Navigate using the sidebar

> To stop the app: click the cell output area and press **I, I** (interrupt kernel), or click the stop button.

In [ ]:
!python -m streamlit run app/main.py

---

## Section B — TikTok Data Scraper

Run these cells to collect real comment data from your TikTok account.
Complete Section A first so all packages are installed.

### Step 4 — Install scraper packages

This installs the TikTokApi library and Playwright browser automation.
Run once.

In [ ]:
!python -m pip install TikTokApi==6.5.2 playwright>=1.40.0

### Step 5 — Download the Chromium browser

Playwright needs a real browser to open TikTok. This downloads a copy of Chromium.
Run once.

In [ ]:
!python -m playwright install chromium

### Step 6 — Run the scraper

**Before running:** If TikTok blocked your login attempts, skip the browser login.
Instead, get your `msToken` manually:
1. Open TikTok in Chrome and log in normally
2. Press F12 → Application tab → Cookies → tiktok.com
3. Find `msToken`, copy the value
4. Create a file called `session.json` in the project folder with:
   ```json
   {"ms_token": "PASTE_YOUR_TOKEN_HERE", "saved_at": "2026-01-01", "username": "ichbinnelo"}
   ```

Then run the cell below. It will collect all your videos and comments into `data/`.

In [ ]:
!python scrape.py

### Step 7 — Check scraped files

In [ ]:
import os
from pathlib import Path

data_folder = Path('./data')
files = [f for f in data_folder.iterdir() if f.suffix == '.csv']

if files:
    print('CSV files in data/ folder:')
    for f in sorted(files):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name}  ({size_kb:.1f} KB)')
else:
    print('No CSV files found yet. Run the scraper in Step 6 first.')

### Step 8 — Preview scraped data

In [ ]:
import pandas as pd
from pathlib import Path

data_folder = Path('./data')

video_files   = sorted(data_folder.glob('videos_*.csv'),   reverse=True)
comment_files = sorted(data_folder.glob('comments_*.csv'), reverse=True)

if not video_files:
    print('No scraped data found yet. Run the scraper in Step 6 first.')
else:
    videos_df   = pd.read_csv(video_files[0],   encoding='utf-8-sig')
    comments_df = pd.read_csv(comment_files[0], encoding='utf-8-sig')

    print(f'Videos:   {len(videos_df)} rows  —  {video_files[0].name}')
    print(f'Comments: {len(comments_df)} rows  —  {comment_files[0].name}')
    print()
    print('--- Videos preview ---')
    display(videos_df.head())
    print('--- Comments preview ---')
    display(comments_df.head())